#### Objetivo del notebook

Validar que los clusters obtenidos mediante KModes representan perfiles diferenciados y establecer cuál de ellos presenta mayor similitud con el perfil de los casos de muerte. Esta validación permitirá asignar etiquetas interpretables a los clusters antes del entrenamiento de modelos supervisados.

Entrada:

Tablas:  

            resultados_kmodes , origen: notebook 03_1
        
            centroides_kmodes , origen: notebook 03_2 
        
            centroide_muerte  , origen: notebook 03_3
        
            distancias_centroides , origen: notebook 03_4
        
            detalle_distancias_centroides , origen: notebook 03_4
        
            nofatales_clusterizados , origen: notebook 03_2
        
            representatividad_muerte , origen: notebook 03_3

Interpretación: a lo largo del notebook.



In [0]:
#Importar librerías
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce

**1. Tabla resumen (A) de los clusters obtenidos de la aplicación de la técnica de agrupamiento K-modes o K-modas, en el grupo de víctimas VIF o VF que siguen vivas o que se han denominado no fatales, con el objeto de establecer si existen clusters heterogéneos entre sí, con los cuales se analice las distancias de estos, al grupo de víctimas VIF o VP que han muerto por violencia de esta naturaleza, al que se ha denominado muerte. con el objeto de etiquetar los grupos no fatales, segun su distancia al grupo muerte.**

In [0]:
#Leer la tabla Delta

tabla = "ml_proyecto_7405607705157039.default.resultados_kmodes"

df_resultados_kmodes = spark.table(tabla)

In [0]:
#Visualizar tabla
display(df_resultados_kmodes)

interpretación: se seleccionaron K= 3 clusters, teniendo en cuenta que la diferencia del costo (disimilitud intra-cluster) es mayor cuando se pasa del cluster 2 al 3. Adicionalmente el balance es de 0.5613, esto quiere decir, que el cluster más pequeño representa 56.13% del más grande, esto garantiza que no se formen agrupaciones muy pequeñas, que no contribuya a definir un perfil. Por otra parte, el algoritmo K-Modes convergió en dos iteraciones. Esto indica que, tras dos ciclos de asignación de los individuos y la actualización de los centroide de los clusters, la composición de los grupos se estabilizó y no se observaron mejoras adicionales en la función de costo. En consecuencia, los tres centroides obtenidos representan la solución estable alcanzada por el algoritmo para esa ejecución.

**2. La primera tabla (B) contiene las características de los centroides de cada uno de los 3 clusters, formado a partir de la base de víctimas no fatales y el uso de la técnica de agrupamiento kmodes, cada cluster etiquetado como 0, 1 y 2. La segunda tabla (C) el centroide del cluster de la base de víctimas de VIF o VP previa, y posteriormente murieron por este tipo de violencia, etiquetado como -1.**     

In [0]:
#Leer tablas Delta

tabla1 = "ml_proyecto_7405607705157039.default.centroides_kmodes"

df_centroides_kmodes = spark.table(tabla1)

tabla2 = "ml_proyecto_7405607705157039.default.centroide_muerte"

df_centroide_muerte = spark.table(tabla2)

In [0]:
#Visualizar tabla1
display(df_centroides_kmodes)

In [0]:
#Visualizar tabla2
display(df_centroide_muerte)

**3. Tabla de las distancias entre centroides (D) y tabla de los detalles de las distancias entre centroides (E)**


In [0]:
#Leer tablas Delta

tabla1 = "ml_proyecto_7405607705157039.default.distancias_centroides"

df_distancias_centroides = spark.table(tabla1)

tabla2 = "ml_proyecto_7405607705157039.default.detalle_distancias_centroides"

df_detalle_distancias_centroides = spark.table(tabla2)

In [0]:
#Visualizar tabla1
display(df_distancias_centroides)

In [0]:
#Visualizar tabla2
display(df_detalle_distancias_centroides)

**4. Tabla (F) del porcentaje que representa cada categoría modal dentro de su cluster**

In [0]:
#Leer tabla que contiene los casos no fatales con su respectiva etiqueta de cluster 0, 1 o 2

df_caso_nofatales_clusterizados = spark.table(
    "ml_proyecto_7405607705157039.default.nofatales_clusterizados"
)

In [0]:
#Variables del modelo
variables = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "presunto_agresor_cod"
]

In [0]:
#Crear lista para almacenar los resultados
resultados = []

In [0]:
#recorrer cada variable

for variable in variables:

    # Conteo por cluster y categoría
    conteos = (
        df_caso_nofatales_clusterizados
        .groupBy("cluster", variable)
        .count()
    )

    # Total de registros por cluster
    totales = (
        df_caso_nofatales_clusterizados
        .groupBy("cluster")
        .agg(F.count("*").alias("total_cluster"))
    )

    # Unir para calcular porcentaje
    conteos = conteos.join(totales, on="cluster")

    conteos = conteos.withColumn(
        "porcentaje",
        F.round(
            100 * F.col("count") / F.col("total_cluster"),
            2
        )
    )

    # Seleccionar únicamente la categoría modal
    ventana = Window.partitionBy("cluster").orderBy(F.desc("count"))

    moda = (
        conteos
        .withColumn(
            "rn",
            F.row_number().over(ventana)
        )
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    moda = (
        moda
        .withColumn("variable", F.lit(variable))
        .withColumnRenamed(variable, "categoria_modal")
        .withColumnRenamed("count", "frecuencia")
    )

    resultados.append(moda)

In [0]:
#Unir todos los resultados

df_representatividad = reduce(
    lambda x, y: x.unionByName(y),
    resultados
)

In [0]:
#Obtener tabla del porcentaje de representatividad de los casos segun las características del centroide
df_representatividad = df_representatividad.select(
    "cluster",
    "variable",
    "categoria_modal",
    "frecuencia",
    "total_cluster",
    "porcentaje"
)

display(df_representatividad.orderBy("cluster", "variable"))

**5. Tabla (G) del porcentaje que representa cada categoría modal dentro del cluster muerte (-1)**

In [0]:
#Leer la tabla Delta

tabla = "ml_proyecto_7405607705157039.default.representatividad_muerte"

df_representatividad_muerte = spark.table(tabla)

In [0]:
df_representatividad_muerte.display()

Interpretación:


-El cluster 0: este cluster tiene una homogeneidad moderada, teniendo en cuenta que la representación del los casos segun las categorías del centroide esta entre 44.25% a 58.69% para las variables sexo, escolaridad, estado civil, mecanismo causal, contexto del hecho y ciclo vital (F). Entre las cuatro primeras variables anteriores, están las coincidentes con el cluster -1 de muerte (D).

Adicionalmente, este es el cluster más cercano al cluster -1 con una distancia de 3 variables diferentes (D). Por lo tanto, este cluster se etiquetará "Peligro Grave". Tamaño del grupo = 167687 casos

-El cluster 2: tiene una homogeneidad alta, teniendo en cuenta que la representación del los casos segun las categorías del centroide esta entre 62.6% a 83.34%, exceptuando la variable mecanismo causal (49.26%)(F).

Adicionalmente difiere en 5 variables con el cluster -1 (D), por lo tanto, se etiquetará con "Peligro Moderado". Tamaño del grupo = 176830 casos

-El cluster 1: tiene una homogeneidad alta, teniendo en cuenta que la representación del los casos segun las categorías del centroide esta entre 50.42% a 88.63% para todas las variables (F). Por otra parte, este cluster es el más lejano al cluster -1, con distancia de 6 variables diferentes (D). Por lo tanto, este cluster se etiquetará "Peligro Bajo".

-los centroides de los clusters 1 y 2 difieren en 3 variables; ciclo vital, estado civil y presunto agresor (D). Tamaño del grupo = 298774 casos

-El cluster -1 que representan los casos donde las victimas mueren, se etiquetará "Peligro extremo", Tamaño del grupo = 101 casos. Este grupo tiene una homogenidad alta, teniendo en cuenta que la representación de los casos segun las categorias del centroide estan entre 55.45% a 76.24% para 5 variables, contexto del hecho (76.24%), estado civil (69.31%), sexo (61.39%), ciclo vital (57.43%) y escolaridad (55.45%) (G). Por otra parte el grupo peligro grave, es mas diverso en relación a las características mecanismo causal (39.6%) y presunto agresor (33.6%). 

Cabe resaltar que al comparar el cluster peligro grave con el alto, moderado y bajo, las 5 variables;  contexto del hecho, estado civil, sexo, ciclo vital y escolaridad, son más determinantes para establecer que un individuo pertenece o no a otro grupo (D).


